# 02 -- Diabetes 130-US Hospitals (1999-2008)

## Research question

Same as dataset 1: how much of an observed subgroup performance gap is
genuine algorithmic bias versus an artifact of prevalence, calibration, and
case mix?

## Dataset

UCI #296, "Diabetes 130-US hospitals for years 1999-2008": 101,766 inpatient
encounters for diabetic patients. Outcome: **30-day readmission**
(`readmitted == "<30"`). Protected attribute: `gender` (Male/Female).
`race` is also available and is used as a covariate block in the case-mix
waterfall.

## A methodological note on the decision threshold

30-day readmission has a base rate of only ~11%. At the default 0.5
threshold, almost nothing is ever classified positive (predicted-positive
rate < 0.3%), which makes PPV/sensitivity essentially undefined-by-tiny-
sample-size. We therefore use `threshold="prevalence"`: the pipeline sets
the decision threshold to the *training-set prevalence* of the outcome,
giving a much more balanced operating point. This is implemented generically
in `pipeline.run_fairness_analysis` so any low-prevalence dataset can opt
into it.

In [ ]:
import sys, os
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
RESULTS_DIR = ROOT / "results"
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.datasets import load_diabetes130
from src.pipeline import run_fairness_analysis
from src import figures as figs

pd.set_option("display.width", 120)

## 1. Load and confirm the data

`datasets.load_diabetes130`:
- drops the 3 rows with `gender == "Unknown/Invalid"`,
- derives `readmit_30d = 1` if `readmitted == "<30"` else 0,
- encodes `age` as a bin-midpoint numeric, the 23 medication columns as
  ordinal dose-intensity (0-3), `max_glu_serum`/`A1Cresult` as ordinal
  severity, `change`/`diabetesMed` as binary, and `race` as one-hot dummies,
- drops high-missingness / identifier columns (`weight`, `payer_code`,
  `medical_specialty`, `encounter_id`, `patient_nbr`, `diag_1-3`).

In [ ]:
d = load_diabetes130()

df = d["df"]
print("Dataset:", d["name"])
print("Shape:", df.shape)
print("Target column:", d["target_col"], " | Group column:", d["group_col"])
print("Group labels:", d["group_labels"])
print("\nNumber of features:", len(d["feature_cols"]))
print("Any nulls in features?", df[d["feature_cols"]].isnull().sum().sum())

In [ ]:
# Subgroup sizes and positive-case counts BEFORE modeling (TRIPOD+AI standard).
target_col, group_col = d["target_col"], d["group_col"]

for val, label in d["group_labels"].items():
    sub = df[df[group_col] == val]
    n_pos = sub[target_col].sum()
    print(f"{label:8s}: n={len(sub):,}  positives={n_pos:,}  prevalence={n_pos/len(sub):.4f}")

print()
print("Race breakdown (used in the case-mix waterfall, not as the primary group):")
# `race` is now a raw categorical source column; its one-hot expansion is
# fitted on training rows only inside the pipeline.
print("Race categories:", sorted(df["race"].dropna().unique()))
print("Encounters:", len(df), "| unique patients:", df["patient_nbr"].nunique())

Both Male (n=5,205 positive) and Female (n=6,152 positive) subgroups are
well above the ~100-positive threshold. Unlike dataset 1, the raw
*prevalence* gap between Male and Female is small here (~11.1% vs 11.3%) --
so we should expect prevalence adjustment to do *less* work on this dataset,
and any gaps that show up are less likely to be prevalence artifacts.

## 2. Run the full pipeline

Group A = Male, Group B = Female; gaps are **(Male) - (Female)**.
`threshold="prevalence"` per the note above.

In [ ]:
result = run_fairness_analysis(
    df=d["df"],
    feature_cols=d["feature_cols"],
    target_col=d["target_col"],
    group_col=d["group_col"],
    group_a_value="Male",
    group_b_value="Female",
    group_labels=d["group_labels"],
    covariate_blocks=d["covariate_blocks"],
    threshold="prevalence",
    test_size=0.3,
    random_state=42,
    n_boot=1000,
    verbose=True,
    # Corrected path: preprocessing is fitted on training rows only,
    # and (where a clustering identifier exists) the split, selection
    # CV, calibration folds and bootstrap are patient-grouped.
    cluster_ids=d["df"]["patient_nbr"],
    preprocess_spec=d["preprocess_spec"],
)

## 3. Reading the results

**Raw gaps (Step 3).** Prevalence is nearly identical between Male and
Female (~11.1% vs 11.3%), but the model is noticeably better at
discriminating in Female patients (AUROC gap ~ -0.012, Male lower).
`sensitivity`/`fnr` show men have a meaningfully *lower* recall and *higher*
FNR for 30-day readmission -- this gap is **not** explained by prevalence,
since prevalence is nearly equal between groups.

**Prevalence adjustment (Step 4).** Because the raw prevalence gap is small
here, prevalence adjustment moves PPV/NPV/predicted-positive-rate gaps only
modestly compared to dataset 1 (where the PPV gap *reversed* dramatically).
Compare the `attenuation_pct` values directly with dataset 1's: this is the
kind of cross-dataset contrast the pooled meta-analysis (Step 3 of the
project) will summarize.

**Bootstrap CIs (Step 5).** The sensitivity/FNR gap being significant *and*
prevalence-adjustment doing little to it is exactly the "residual inequity"
signature the project hypothesis calls out -- two independent datasets now
show a male-disadvantaged recall gap that survives prevalence adjustment.

**Calibration adjustment (Step 6).** Equal-sensitivity thresholds nearly
zero out the `equal_opportunity_diff` by construction, but watch what
happens to `disparate_impact_ratio` and `ppv_diff` as a result -- this is
the fairness trade-off the calibration step is meant to surface.

**Case-mix waterfall (Step 7).** This regresses `readmit_30d ~ gender +
covariates` (NOT the model's predictions) to see how much of the raw
gender/outcome association survives after adjusting for demographics,
comorbidity count, hospital utilization, medications, and admission/
discharge administrative codes. A `pct_of_raw_gap_explained` > 100% or
negative means the block makes the gender effect on the outcome *larger*,
not smaller -- worth flagging rather than smoothing over.

## 4. Figures

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

fig1 = figs.plot_calibration_curve(
    result["y_test"], result["prob"], result["g_test"],
    group_labels={"Male": "Male", "Female": "Female"},
    title="Calibration by sex (calibrated model) -- Diabetes 130-US Hospitals",
)
fig1.savefig(RESULTS_DIR / "02_diabetes130_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    gap_dict = {
        "Raw": (raw["point"], raw["ci_low"], raw["ci_high"]),
        "Prevalence-adjusted": (adjd["point"], adjd["ci_low"], adjd["ci_high"]),
    }
    fig = figs.plot_gap_bars(
        gap_dict,
        title=f"{key}: raw vs. prevalence-adjusted gap (Male - Female)",
        ylabel="Gap (Male - Female)",
    )
    fig.savefig(RESULTS_DIR / f"02_diabetes130_gap_{key}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
fig_wf = figs.plot_case_mix_waterfall(
    result["case_mix"],
    title="Diabetes 130-US Hospitals: case-mix waterfall (Male vs Female outcome gap)",
)
fig_wf.savefig(RESULTS_DIR / "02_diabetes130_case_mix_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save summary table for cross-dataset pooling

In [ ]:
rows = []
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    pa = result["prevalence_adjustment"][key]
    rows.append({
        "dataset": "diabetes_130_hospitals",
        "metric": key,
        "raw_gap": raw["point"],
        "raw_ci_low": raw["ci_low"],
        "raw_ci_high": raw["ci_high"],
        "prevalence_adjusted_gap": adjd["point"],
        "prevalence_adjusted_ci_low": adjd["ci_low"],
        "prevalence_adjusted_ci_high": adjd["ci_high"],
        "attenuation_pct": pa["attenuation_pct"],
    })

for key in ("sensitivity", "fnr"):
    raw = result["bootstrap_raw"][key]
    rows.append({
        "dataset": "diabetes_130_hospitals",
        "metric": key,
        "raw_gap": raw["point"],
        "raw_ci_low": raw["ci_low"],
        "raw_ci_high": raw["ci_high"],
        "prevalence_adjusted_gap": np.nan,
        "prevalence_adjusted_ci_low": np.nan,
        "prevalence_adjusted_ci_high": np.nan,
        "attenuation_pct": np.nan,
    })

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_DIR / "02_diabetes130_summary.csv", index=False)
summary

In [ ]:
# Frozen held-out outputs.
#
# Persist the row-level held-out predictions, the overall and per-subgroup
# discrimination/calibration table, and the fitted preprocessing + estimator +
# calibration objects. Nothing here changes the analysis: every value written
# is read from `result`, which was produced above. These artifacts let any
# later evaluation be answered without refitting a model.
from src import frozen_outputs as fo

frozen_manifest = fo.write_comparison_artifacts(
    result,
    slug="sex_diabetes130",
    dataset=d["name"],
    comparison_type="sex",
    comparison="Male vs Female",
    results_dir=RESULTS_DIR,
    cluster_ids=d["df"]["patient_nbr"],
)
print("frozen predictions :", frozen_manifest["predictions"])
print("frozen performance :", frozen_manifest["performance"])
print("serialized objects :", frozen_manifest["model"]["status"])
